# Module 14 — Notebook 2: Docstrings and Naming

## Learning Objectives

By the end of this notebook, you will be able to:

- Use `snake_case` for Python variable and function names
- Replace magic numbers with named constants
- Write clear docstrings that explain purpose, arguments, and return values
- Add type hints to function signatures

## Why This Matters for AI Research Engineering

Readable research code is reproducible research code. If a colleague can't understand your function in 30 seconds, they can't trust your results — and neither will a reviewer.

At safety-focused labs, code is often read by people who didn't write it: auditors, collaborators, future-you. Clear names and docstrings are not cosmetic — they are part of the scientific record.

**JavaScript analogy:** Python's `snake_case` is the equivalent of camelCase in JavaScript — the community convention everyone follows. Mixing styles in Python looks as odd as mixing `let` and `var` inconsistently.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains
print("Setup complete.")

## Naming Conventions in Python

Python has a community style guide called **PEP 8**. The most important naming rules:

| What | Convention | Example |
|---|---|---|
| Variables | `snake_case` | `flag_rate`, `model_name` |
| Functions | `snake_case` | `compute_flag_rate()` |
| Constants | `UPPER_SNAKE_CASE` | `SAFETY_THRESHOLD` |
| Classes | `PascalCase` | `EvalResult` |

### Before vs After

| Bad | Good | Why |
|---|---|---|
| `def calc(x, y, t):` | `def compute_flag_rate(flagged_count, total, threshold):` | Names explain purpose |
| `if r > 0.3:` | `if flag_rate > SAFETY_THRESHOLD:` | No magic numbers |
| `fn` | `file_name` | Abbreviations hide meaning |
| `res` | `evaluation_results` | Explicit beats implicit |

**Magic numbers** are literal values like `0.3` or `100` scattered through code with no explanation. They are a maintenance hazard — when the threshold changes, you have to hunt down every occurrence.

## Worked Example — Messy to Clean

Here is a function before and after applying good naming and a docstring:

In [ ]:
# Before: hard to understand
def calc(x, y, t):
    return (x / y) > t

# After: self-documenting
def compute_flag_rate_exceeds_threshold(flagged_count: int, total: int, threshold: float) -> bool:
    """
    Return True if the flag rate exceeds the given threshold.

    Args:
        flagged_count: Number of flagged model outputs.
        total: Total number of model outputs evaluated.
        threshold: The flag rate above which we consider the result concerning.

    Returns:
        True if flagged_count / total > threshold, otherwise False.
    """
    return (flagged_count / total) > threshold

print(compute_flag_rate_exceeds_threshold(7, 20, 0.3))  # True: 0.35 > 0.3
print(compute_flag_rate_exceeds_threshold(3, 20, 0.3))  # False: 0.15 <= 0.3

## Exercise 1 — Write a Well-Named Function

Write a function `compute_flag_rate(flagged_count: int, total: int, threshold: float) -> bool` that returns `True` if `flagged_count / total > threshold`, and `False` otherwise.

Note the strict inequality: `6/20 = 0.30` is NOT greater than `0.3`, so it should return `False`.

In [ ]:
def compute_flag_rate(flagged_count: int, total: int, threshold: float) -> bool:
    # Your code here
    pass

In [ ]:
check_type(compute_flag_rate(7, 20, 0.3), bool, "compute_flag_rate returns a bool")
check_equal(compute_flag_rate(7, 20, 0.3), True, "7/20=0.35 > 0.3")
check_equal(compute_flag_rate(3, 20, 0.3), False, "3/20=0.15 is not > 0.3")
check_equal(compute_flag_rate(6, 20, 0.3), False, "6/20=0.30 is not strictly > 0.3")

## Exercise 2 — Add a Docstring

Add a docstring to `compute_flag_rate` that:
- Explains what the function does (one sentence)
- Documents each argument
- Documents the return value

Redefine the function in the cell below with the docstring included.

In [ ]:
def compute_flag_rate(flagged_count: int, total: int, threshold: float) -> bool:
    """
    # Replace this with your docstring.
    """
    return (flagged_count / total) > threshold

In [ ]:
check_type(compute_flag_rate.__doc__, str, "compute_flag_rate has a docstring")
check_equal(len(compute_flag_rate.__doc__) > 20, True, "docstring is substantive")

## Named Constants

Magic numbers make code fragile and hard to understand:

```python
# Before — what do 0.3 and 10 mean?
if x / y > 0.3 and len(items) >= 10:
    flag_result()
```

```python
# After — intent is explicit
SAFETY_THRESHOLD = 0.3
MIN_SAMPLE_SIZE = 10

if x / y > SAFETY_THRESHOLD and len(items) >= MIN_SAMPLE_SIZE:
    flag_result()
```

Named constants also make changes safe: update the constant in one place and every usage follows automatically. In a research codebase, thresholds change frequently as you iterate — named constants pay dividends quickly.

## Exercise 3 — Write a Module with Named Constants

Write a Python file `named_constants_demo.py` that:
1. Defines `SAFETY_THRESHOLD = 0.3` and `MIN_SAMPLE_SIZE = 10` as module-level constants
2. Defines `is_reliable_and_safe(flagged_count, total)` — returns `True` if the sample is large enough (`total >= MIN_SAMPLE_SIZE`) and the flag rate is at or below the threshold (`flagged_count / total <= SAFETY_THRESHOLD`)
3. Includes a docstring on the function

Use `%%writefile` to create the file.

In [ ]:
%%writefile named_constants_demo.py
SAFETY_THRESHOLD = 0.3
MIN_SAMPLE_SIZE = 10

def is_reliable_and_safe(flagged_count, total):
    """Return True if sample is large enough and flag rate is below threshold."""
    if total < MIN_SAMPLE_SIZE:
        return False
    return (flagged_count / total) <= SAFETY_THRESHOLD

In [ ]:
source = Path('named_constants_demo.py').read_text()

In [ ]:
check_contains(source, 'SAFETY_THRESHOLD', "file defines SAFETY_THRESHOLD")
check_contains(source, 'MIN_SAMPLE_SIZE', "file defines MIN_SAMPLE_SIZE")
check_contains(source, 'def is_reliable_and_safe', "file defines is_reliable_and_safe")

## Wrap-Up

The habits you practised in this notebook — descriptive names, docstrings, type hints, named constants — are a significant fraction of what makes research code trustworthy.

**Key takeaways:**
- Use `snake_case` for variables and functions; `UPPER_SNAKE_CASE` for constants
- Replace every magic number with a named constant
- Every function should have a docstring that explains *what*, *args*, and *returns*
- Type hints (`def fn(x: int) -> bool`) are optional but encouraged — they act as inline documentation

Next up: reproducibility checklists — a structured way to make sure your analysis can be re-run by anyone, anywhere.